In [1]:
import os

# Fix: TensorFlow import crashes in this environment with protobuf ("MessageFactory.GetPrototype").
# We force a stable, end-to-end run by disabling TF usage entirely and relying on the existing
# non-TF baseline path (which we will keep identical in modeling approach: per-patient linear trend).
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")

import random
import numpy as np
import pandas as pd

random.seed(42)
np.random.seed(42)

TF_AVAILABLE = False
TF_IMPORT_ERROR = RuntimeError(
    "TensorFlow disabled due to protobuf incompatibility in this environment."
)



In [2]:
MIN_MAX = {
    "Weeks": (-5.0, 133.0),
    "FVC": (827.0, 6399.0),
    "Percent": (28.877577, 153.145378),
    "Age": (49.0, 88.0),
    "typical_fvc": (827.0, 6399.0),
}



In [3]:
IMG_SIZE = 256
NUM_OF_SCANS = 10
BATCH_SIZE = 32

TEST_DF = pd.read_csv("../input/osic-pulmonary-fibrosis-progression/test.csv")
SAMPLE_SUBMISSION = pd.read_csv(
    "../input/osic-pulmonary-fibrosis-progression/sample_submission.csv"
)

model_path = "../input/osic-resnet-v3"
if os.path.isdir(model_path):
    all_files = [os.path.join(model_path, x) for x in os.listdir(model_path)]
    model_weights = sorted(
        [
            w
            for w in all_files
            if os.path.isfile(w) and (w.endswith(".h5") or w.endswith(".ckpt"))
        ]
    )
else:
    model_weights = []  # fallback

training_features = [
    "Weeks",
    "min_week",
    "min_week_FVC",
    "typical_fvc",
    "Age",
    "Male",
    "Female",
    "Never smoked",
    "Currently smokes",
    "Ex-smoker",
]
num_of_features = len(training_features)




In [4]:
def create_typical_fvc(df):
    df = df.copy()
    df["typical_fvc"] = df["FVC"] / df["Percent"] * 100.0
    return df


def normalize(df):
    df = df.copy()
    for feature, min_max in MIN_MAX.items():
        df[feature] = (df[feature] - min_max[0]) / (min_max[1] - min_max[0])
    return df


TEST_DF = create_typical_fvc(TEST_DF)

TEST_DF["min_week"] = np.zeros(len(TEST_DF), dtype="float32")
TEST_DF["min_week_FVC"] = np.zeros(len(TEST_DF), dtype="float32")

TEST_DF["Never smoked"] = (TEST_DF["SmokingStatus"] == "Never smoked").astype("uint8")
TEST_DF["Currently smokes"] = (TEST_DF["SmokingStatus"] == "Currently smokes").astype(
    "uint8"
)
TEST_DF["Ex-smoker"] = (TEST_DF["SmokingStatus"] == "Ex-smoker").astype("uint8")

TEST_DF["Male"] = (TEST_DF["Sex"] == "Male").astype("uint8")
TEST_DF["Female"] = (TEST_DF["Sex"] == "Female").astype("uint8")

TEST_DF = normalize(TEST_DF)

for patient in np.unique(TEST_DF["Patient"]):
    m = TEST_DF["Patient"] == patient
    TEST_DF.loc[m, "min_week"] = TEST_DF.loc[m, "Weeks"].min()
    TEST_DF.loc[m, "min_week_FVC"] = TEST_DF.loc[m, "FVC"].values[0]

TEST_DF.head()



/tmp/ipykernel_11/3529694359.py:32: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.036231884057971016' has dtype incompatible with float32, please explicitly cast to a compatible dtype first.
  TEST_DF.loc[m, "min_week"] = TEST_DF.loc[m, "Weeks"].min()
/tmp/ipykernel_11/3529694359.py:33: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.5348169418521177' has dtype incompatible with float32, please explicitly cast to a compatible dtype first.
  TEST_DF.loc[m, "min_week_FVC"] = TEST_DF.loc[m, "FVC"].values[0]


,Patient,Weeks,FVC,Percent,Age,Sex,SmokingStatus,typical_fvc,min_week,min_week_FVC,Never smoked,Currently smokes,Ex-smoker,Male,Female
0,ID00014637202177757139317,0.036232,0.534817,0.492477,0.179487,Male,Ex-smoker,0.610086,0.036232,0.534817,0,0,1,1,0
1,ID00019637202178323708467,0.130435,0.228464,0.514865,0.871795,Female,Ex-smoker,0.257448,0.130435,0.228464,0,0,1,0,1
2,ID00047637202184938901501,0.050725,0.446159,0.491293,0.487179,Male,Ex-smoker,0.512742,0.050725,0.446159,0,0,1,1,0
3,ID00082637202201836229724,0.173913,0.375269,0.570681,0.000000,Female,Currently smokes,0.376346,0.173913,0.375269,0,1,0,0,1
4,ID00126637202218610655908,0.166667,0.277818,0.245417,0.743590,Male,Ex-smoker,0.569454,0.166667,0.277818,0,0,1,1,0


In [5]:
# TF branch disabled; keep placeholders so later cells remain in original order.
volumes = {}


class Dataset(object):
    def __init__(self, batch_size=BATCH_SIZE, mode=0):
        self.indices = np.arange(0, len(SAMPLE_SUBMISSION), 1)
        self.batch_size = batch_size
        self.mode = mode  # 0 - Training, 1 - Test

    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))




In [6]:
# TF model-building code intentionally not executed in this environment.
build_model = None



In [7]:
models = []



In [8]:
predictions = None




In [9]:
def denormalize(y):
    return y * (MIN_MAX["FVC"][1] - MIN_MAX["FVC"][0]) + MIN_MAX["FVC"][0]


if TF_AVAILABLE and predictions is not None:
    # Unused in this environment; kept to preserve original structure/semantics when TF is available.
    predictions = predictions / len(models)
    predictions = denormalize(predictions)

    predictions = np.nan_to_num(
        predictions,
        nan=MIN_MAX["FVC"][0],
        posinf=MIN_MAX["FVC"][1],
        neginf=MIN_MAX["FVC"][0],
    )
    predictions = np.clip(predictions, MIN_MAX["FVC"][0], MIN_MAX["FVC"][1])

    FVC = predictions[:, 1]

    raw_conf = predictions[:, 2] - predictions[:, 0]
    raw_conf = np.maximum(raw_conf, 1.0)

    base = float(np.percentile(raw_conf, 75))
    scale = 200.0 / base if base > 0 else 1.0
    Confidence = raw_conf * scale
    Confidence = np.maximum(Confidence, 70.0)
    Confidence = np.minimum(Confidence, 1000.0)

    SAMPLE_SUBMISSION["FVC"] = np.round(FVC).astype(int)
    SAMPLE_SUBMISSION["Confidence"] = np.round(Confidence).astype(int)

else:
    train_df = pd.read_csv("../input/osic-pulmonary-fibrosis-progression/train.csv")

    # Compute per-patient slope (ml/week) as before (core linear-trend logic preserved)
    slopes = []
    patient_slope = {}
    for pid, g in train_df.groupby("Patient"):
        if g.shape[0] >= 2:
            x = g["Weeks"].values.astype(np.float64)
            y = g["FVC"].values.astype(np.float64)
            xv = x - x.mean()
            denom = np.sum(xv * xv)
            if denom > 0:
                slope = float(np.sum(xv * (y - y.mean())) / denom)
                slopes.append(slope)
                patient_slope[pid] = slope

    global_slope = float(np.median(slopes)) if len(slopes) else -10.0

    # Fix (score-moving toward target): estimate typical absolute FVC residual (ml) from train,
    # instead of using slope dispersion (ml/week) as if it were an FVC residual.
    abs_resids = []
    for pid, g in train_df.groupby("Patient"):
        if g.shape[0] >= 2:
            x = g["Weeks"].values.astype(np.float64)
            y = g["FVC"].values.astype(np.float64)
            xv = x - x.mean()
            denom = np.sum(xv * xv)
            if denom > 0:
                slope = float(np.sum(xv * (y - y.mean())) / denom)
                intercept = float(y.mean() - slope * x.mean())
                yhat = slope * x + intercept
                abs_resids.extend(np.abs(y - yhat).tolist())
    global_abs_resid_ml = float(np.median(abs_resids)) if len(abs_resids) else 200.0

    test_base = pd.read_csv("../input/osic-pulmonary-fibrosis-progression/test.csv")
    base_fvc = dict(zip(test_base["Patient"].values, test_base["FVC"].values))
    base_week = dict(zip(test_base["Patient"].values, test_base["Weeks"].values))

    pw = SAMPLE_SUBMISSION["Patient_Week"].values
    patients = np.array([x.split("_")[0] for x in pw])
    weeks = np.array([float(x.split("_")[1]) for x in pw], dtype=np.float64)

    fvc_pred = np.zeros(len(pw), dtype=np.float64)
    conf_pred = np.zeros(len(pw), dtype=np.float64)

    for i, pid in enumerate(patients):
        w = weeks[i]
        b_fvc = float(base_fvc.get(pid, 2500.0))
        b_w = float(base_week.get(pid, 0.0))
        slope = float(patient_slope.get(pid, global_slope))
        fvc = b_fvc + slope * (w - b_w)

        horizon = abs(w - b_w)

        # Keep same overall form: base + horizon term + global-noise term, but with correct units.
        sigma = 100.0 + 2.0 * horizon + 1.5 * global_abs_resid_ml
        sigma = float(np.clip(sigma, 70.0, 1000.0))

        fvc_pred[i] = fvc
        conf_pred[i] = sigma

    fvc_pred = np.nan_to_num(
        fvc_pred,
        nan=MIN_MAX["FVC"][0],
        posinf=MIN_MAX["FVC"][1],
        neginf=MIN_MAX["FVC"][0],
    )
    conf_pred = np.nan_to_num(conf_pred, nan=200.0, posinf=1000.0, neginf=70.0)

    fvc_pred = np.clip(fvc_pred, MIN_MAX["FVC"][0], MIN_MAX["FVC"][1])
    conf_pred = np.clip(conf_pred, 70.0, 1000.0)

    SAMPLE_SUBMISSION["FVC"] = np.round(fvc_pred).astype(int)
    SAMPLE_SUBMISSION["Confidence"] = np.round(conf_pred).astype(int)

SAMPLE_SUBMISSION = SAMPLE_SUBMISSION[["Patient_Week", "FVC", "Confidence"]]
SAMPLE_SUBMISSION.to_csv("submission.csv", index=False)

SAMPLE_SUBMISSION.head()

,Patient_Week,FVC,Confidence
0,ID00126637202218610655908_-3,2451,230
1,ID00126637202218610655908_-2,2448,228
2,ID00126637202218610655908_-1,2444,226
3,ID00126637202218610655908_0,2440,224
4,ID00126637202218610655908_1,2437,222
